In [1]:
import sqlite3
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

# 1. Load the CSV and give columns short, readable names
df = pd.read_csv("/content/drive/MyDrive/cs-training.csv", index_col=0)
df.index.name = "id"
df.columns = ["defaulted", "utilization", "age", "late_30_59", "debt_ratio",
              "monthly_income", "open_lines", "late_90", "real_estate_loans",
              "late_60_89", "dependents"]

# 2. Put it into a SQL database as a table called "borrowers"
conn = sqlite3.connect("credit.db")
df.to_sql("borrowers", conn, if_exists="replace")

# 3. Helper: run any SQL query and show the result as a table
def q(sql):
    return pd.read_sql(sql, conn)

# 4. First queries
print(q("""
SELECT COUNT(*)                        AS customers,
       SUM(defaulted)                  AS defaulters,
       ROUND(AVG(defaulted) * 100, 2)  AS default_rate_pct
FROM borrowers
""").to_string(index=False))

print(q("""
SELECT SUM(monthly_income IS NULL)  AS missing_income,
       SUM(dependents IS NULL)      AS missing_dependents,
       SUM(age = 0)                 AS age_zero,
       SUM(late_90 IN (96, 98))     AS late_error_codes,
       SUM(utilization > 1)         AS utilization_over_100pct
FROM borrowers
""").to_string(index=False))

Mounted at /content/drive
 customers  defaulters  default_rate_pct
    150000       10026              6.68
 missing_income  missing_dependents  age_zero  late_error_codes  utilization_over_100pct
          29731                3924         1               269                     3321


In [2]:
conn.executescript("""
DROP VIEW IF EXISTS clean;
CREATE VIEW clean AS
SELECT
  id,
  defaulted,
  age,
  MIN(utilization, 1.5) AS utilization,
  CASE WHEN late_30_59 >= 96 THEN 0 ELSE late_30_59 END AS late_30_59,
  CASE WHEN late_60_89 >= 96 THEN 0 ELSE late_60_89 END AS late_60_89,
  CASE WHEN late_90    >= 96 THEN 0 ELSE late_90    END AS late_90,
  monthly_income,
  monthly_income IS NULL AS income_missing,
  COALESCE(dependents, 0) AS dependents,
  open_lines,
  real_estate_loans
FROM borrowers
WHERE age > 0;
""")

print(q("""
SELECT COUNT(*)          AS customers,
       MAX(late_90)      AS max_late_90,
       MAX(utilization)  AS max_utilization,
       SUM(income_missing) AS income_missing_flagged
FROM clean
""").to_string(index=False))

 customers  max_late_90  max_utilization  income_missing_flagged
    149999           17              1.5                   29731


In [3]:
# Query A: default rate by age band
print(q("""
SELECT
  CASE WHEN age < 30 THEN '1. Under 30'
       WHEN age < 40 THEN '2. 30-39'
       WHEN age < 50 THEN '3. 40-49'
       WHEN age < 60 THEN '4. 50-59'
       ELSE '5. 60+' END              AS age_band,
  COUNT(*)                            AS customers,
  SUM(defaulted)                      AS defaulters,
  ROUND(100.0 * AVG(defaulted), 1)    AS default_rate_pct
FROM clean
GROUP BY age_band
ORDER BY age_band
""").to_string(index=False))

# Query B: default rate by income group (5 equal groups) plus missing income
print(q("""
WITH ranked AS (
  SELECT defaulted, monthly_income,
         NTILE(5) OVER (ORDER BY monthly_income) AS income_group
  FROM clean
  WHERE monthly_income IS NOT NULL
)
SELECT income_group,
       COUNT(*)                          AS customers,
       MIN(monthly_income)               AS min_income,
       MAX(monthly_income)               AS max_income,
       ROUND(100.0 * AVG(defaulted), 1)  AS default_rate_pct
FROM ranked
GROUP BY income_group

UNION ALL

SELECT 'missing', COUNT(*), NULL, NULL, ROUND(100.0 * AVG(defaulted), 1)
FROM clean
WHERE monthly_income IS NULL
""").to_string(index=False))

   age_band  customers  defaulters  default_rate_pct
1. Under 30       8820        1035              11.7
   2. 30-39      23183        2335              10.1
   3. 40-49      34377        2878               8.4
   4. 50-59      35301        2278               6.5
     5. 60+      48318        1500               3.1
income_group  customers  min_income  max_income  default_rate_pct
           1      24054         0.0      3000.0               9.0
           2      24054      3000.0      4544.0               8.6
           3      24054      4545.0      6300.0               7.0
           4      24053      6300.0      9083.0               5.5
           5      24053      9083.0   3008750.0               4.5
     missing      29731         NaN         NaN               5.6


In [4]:
print(q("""
WITH seg AS (
  SELECT
    CASE WHEN utilization < 0.3 THEN '1. Low (<30%)'
         WHEN utilization < 0.7 THEN '2. Medium (30-70%)'
         ELSE '3. High (70%+)' END                         AS utilization_band,
    CASE WHEN late_30_59 + late_60_89 + late_90 = 0 THEN 'Never late'
         WHEN late_60_89 + late_90 = 0             THEN 'Only 30-59 days late'
         ELSE '60+ days late' END                          AS delinquency,
    defaulted
  FROM clean
)
SELECT utilization_band,
       delinquency,
       COUNT(*)                                                    AS customers,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)          AS pct_of_customers,
       ROUND(100.0 * AVG(defaulted), 1)                            AS default_rate_pct,
       ROUND(100.0 * SUM(defaulted) / SUM(SUM(defaulted)) OVER (), 1) AS pct_of_all_defaults
FROM seg
GROUP BY utilization_band, delinquency
ORDER BY default_rate_pct DESC
""").to_string(index=False))

  utilization_band          delinquency  customers  pct_of_customers  default_rate_pct  pct_of_all_defaults
    3. High (70%+)        60+ days late       7386               4.9              44.2                 32.5
2. Medium (30-70%)        60+ days late       2389               1.6              29.0                  6.9
    3. High (70%+) Only 30-59 days late       5163               3.4              21.9                 11.3
     1. Low (<30%)        60+ days late       3105               2.1              19.3                  6.0
2. Medium (30-70%) Only 30-59 days late       4288               2.9              10.9                  4.6
    3. High (70%+)           Never late      17398              11.6               9.0                 15.6
     1. Low (<30%) Only 30-59 days late       7762               5.2               6.0                  4.6
2. Medium (30-70%)           Never late      20493              13.7               4.2                  8.5
     1. Low (<30%)          

In [5]:
import pandas as pd
pd.set_option("display.width", 200)

# Run this first to see your real names:
print(pd.read_sql("PRAGMA table_info(clean)", conn)[["name"]].T)

C = dict(
    dq="dq",            # default flag (0/1)
    util="util",        # revolving utilisation (capped at 1.5 in the view)
    l30="late30",       # 30-59 days late count
    l60="late60",       # 60-89 days late count
    l90="late90",       # 90+ days late count
)
def q(sql): return pd.read_sql(sql.format(**C), conn)

      0          1    2            3           4           5        6               7               8           9           10                 11
name  id  defaulted  age  utilization  late_30_59  late_60_89  late_90  monthly_income  income_missing  dependents  open_lines  real_estate_loans


In [7]:
cols = pd.read_sql("PRAGMA table_info(clean)", conn)["name"].tolist()
print(cols)

def find(*keys):
    for c in cols:
        if all(k in c.lower() for k in keys):
            return c
    return None

C = dict(
    dq   = find("dlq") or find("default") or find("target") or find("serious"),
    util = find("util"),
    l30  = find("30"),
    l60  = find("60"),
    l90  = find("90"),
)
print(C)
assert all(C.values()), "One is None above: set it by hand from the printed column list"

def q(sql): return pd.read_sql(sql.format(**C), conn)

['id', 'defaulted', 'age', 'utilization', 'late_30_59', 'late_60_89', 'late_90', 'monthly_income', 'income_missing', 'dependents', 'open_lines', 'real_estate_loans']
{'dq': 'defaulted', 'util': 'utilization', 'l30': 'late_30_59', 'l60': 'late_60_89', 'l90': 'late_90'}


In [8]:
dec = q("""
WITH d AS (
  SELECT {dq} AS dq, {util} AS util,
         NTILE(10) OVER (ORDER BY {util}) AS util_decile
  FROM clean
),
agg AS (
  SELECT util_decile,
         COUNT(*) AS customers,
         SUM(dq) AS defaulters,
         ROUND(100.0*SUM(dq)/COUNT(*),2) AS default_rate_pct
  FROM d GROUP BY util_decile
)
SELECT util_decile, customers, defaulters, default_rate_pct,
       RANK() OVER (ORDER BY default_rate_pct DESC) AS risk_rank,
       ROUND(default_rate_pct - LAG(default_rate_pct) OVER (ORDER BY util_decile),2) AS change_vs_prev_decile,
       SUM(defaulters) OVER (ORDER BY util_decile DESC) AS cum_defaulters_from_top,
       ROUND(100.0*SUM(defaulters) OVER (ORDER BY util_decile DESC)
             / SUM(defaulters) OVER (),1) AS cum_pct_of_defaults_captured
FROM agg
ORDER BY util_decile DESC;
""")
dec

,util_decile,customers,defaulters,default_rate_pct,risk_rank,change_vs_prev_decile,cum_defaulters_from_top,cum_pct_of_defaults_captured
0,10,14999,3475,23.17,1,6.58,3475,34.7
1,9,15000,2488,16.59,2,7.79,5963,59.5
2,8,15000,1320,8.80,3,3.55,7283,72.6
3,7,15000,787,5.25,4,1.78,8070,80.5
4,6,15000,521,3.47,5,1.10,8591,85.7
5,5,15000,356,2.37,7,0.46,8947,89.2
6,4,15000,287,1.91,8,0.51,9234,92.1
7,3,15000,210,1.40,9,0.04,9444,94.2
8,2,15000,204,1.36,10,-1.16,9648,96.2
9,1,15000,378,2.52,6,NaN,10026,100.0


In [9]:
q("""
SELECT CASE WHEN {util} = 0 THEN 'zero util' ELSE 'near zero' END AS grp,
       COUNT(*) AS customers,
       ROUND(100.0*SUM({dq})/COUNT(*),2) AS default_rate_pct,
       ROUND(100.0*AVG(CASE WHEN {l30}+{l60}+{l90}>0 THEN 1 ELSE 0 END),1) AS pct_with_lates
FROM clean
WHERE {util} <= (SELECT MAX({util}) FROM (SELECT {util}, NTILE(10) OVER (ORDER BY {util}) d FROM clean) WHERE d = 1)
GROUP BY grp
""")

,grp,customers,default_rate_pct,pct_with_lates
0,near zero,4122,1.41,9.9
1,zero util,10878,2.94,13.2
